Author : Ong Gim Shin
# Feature preprocessing to predict remaining length of stay (LoS)
- The approach is to use vital signs readings from (t0 to tn) in order to predict the bed occupancy rate at tn+m
- The bed occupancy at tn+m will be the patients that are still in the hospital at tn+m **+** admissions **-** discharge
- In order to predict the patients that are still hospitalised on tn+m we, we will predict the length of stay at tn that is extended for >= m days
- The target is then **remaining los** as of the chartdate, instead of **los**

## 1. Transform the vital signs measurements in the data

### Set Environment (Run this before first and you can skip to the portion where the files are read)

In [1]:
import pandas as pd
import numpy as np
import random
import os
from tqdm import tqdm
import miceforest as mf

random.seed(5188)

input_folder = 'Dataset/'
output_folder = 'Merged_Dataset/'
os.makedirs(os.path.dirname(output_folder), exist_ok=True)
pd.set_option('display.max_rows', 100)

### Read CSV from Merged_Datasets folder

In [2]:
sc = pd.read_csv(output_folder + 'selected_chartevents.csv') 
icup = pd.read_csv(output_folder + 'icu_patients.csv') 

In [3]:
sc.label.value_counts()

label
Heart Rate                          711809
Respiratory Rate                    707435
Temperature                         634208
Heart rate Alarm - High             530906
Non Invasive Blood Pressure mean    530100
Resp Alarm - High                   520924
Resp Alarm - Low                    520834
if_Access Lines - Invasive          431301
Inspired O2 Fraction                327343
Arterial Blood Pressure mean        251479
Weight                              248112
High risk (>51) interventions       192314
if_Dialysis                          59977
Height                               43623
ART BP Mean                          33813
if_IABP                               4843
if_Cardiovascular                     1816
if_Impella                            1800
Family Meeting                        1638
Low risk (25-50) interventions         951
Name: count, dtype: int64

In [4]:
sc_icup_joined = pd.merge(sc, icup, on=['subject_id', 'hadm_id', 'stay_id'], how='left')
sc_icup_joined.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'chartdate', 'intime_x', 'label',
       'param_type', 'stay_days', 'value', 'first_careunit', 'intime_y',
       'outtime', 'los', 'gender', 'anchor_age', 'anchor_year',
       'anchor_year_group', 'admission_type', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'hospital_expire_flag', 'icu_types', 'assigned_intime',
       'assigned_outtime'],
      dtype='object')

In [5]:
sc_icup_joined = sc_icup_joined.drop(columns=['intime_y','param_type','discharge_location']).rename(columns={'intime_x':'intime'})
sc_icup_joined = sc_icup_joined[sc_icup_joined.hospital_expire_flag == 0].drop(columns = ['hospital_expire_flag'])

### Obtain the correct age = admit year - anchor year + anchor age

In [6]:
sc_icup_joined.intime = pd.to_datetime(sc_icup_joined.intime)
sc_icup_joined['age'] = sc_icup_joined.intime.dt.year - sc_icup_joined.anchor_year + sc_icup_joined.anchor_age

In [7]:
sc_icup_joined.chartdate = pd.to_datetime(sc_icup_joined.chartdate)
sc_icup_joined.outtime = pd.to_datetime(sc_icup_joined.outtime)
sc_icup_joined = sc_icup_joined.drop(columns=['anchor_age'])

In [8]:
sc_icup_joined = sc_icup_joined[(sc_icup_joined.chartdate >= sc_icup_joined.intime) & (sc_icup_joined.chartdate <= sc_icup_joined.outtime)]

In [9]:
sc_icup_joined.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'chartdate', 'intime', 'label',
       'stay_days', 'value', 'first_careunit', 'outtime', 'los', 'gender',
       'anchor_year', 'anchor_year_group', 'admission_type',
       'admission_location', 'insurance', 'language', 'marital_status', 'race',
       'icu_types', 'assigned_intime', 'assigned_outtime', 'age'],
      dtype='object')

### Pivot the readings in label to columns

In [10]:
pivot = pd.pivot_table(sc_icup_joined, values='value', index=['subject_id', 'hadm_id', 'stay_id','chartdate',
        'intime', 'first_careunit', 'outtime', 'los', 'gender', 'age','stay_days',
        'anchor_year', 'anchor_year_group', 'admission_type',
        'admission_location', 'insurance', 'language', 'marital_status', 'race',
        'icu_types', 'assigned_intime', 'assigned_outtime'], columns=['label'], aggfunc="mean").reset_index()

In [11]:
pivot.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'chartdate', 'intime',
       'first_careunit', 'outtime', 'los', 'gender', 'age', 'stay_days',
       'anchor_year', 'anchor_year_group', 'admission_type',
       'admission_location', 'insurance', 'language', 'marital_status', 'race',
       'icu_types', 'assigned_intime', 'assigned_outtime', 'ART BP Mean',
       'Arterial Blood Pressure mean', 'Family Meeting', 'Heart Rate',
       'Heart rate Alarm - High', 'Height', 'High risk (>51) interventions',
       'Inspired O2 Fraction', 'Low risk (25-50) interventions',
       'Non Invasive Blood Pressure mean', 'Resp Alarm - High',
       'Resp Alarm - Low', 'Respiratory Rate', 'Temperature', 'Weight',
       'if_Access Lines - Invasive', 'if_Cardiovascular', 'if_Dialysis',
       'if_IABP', 'if_Impella'],
      dtype='object', name='label')

In [12]:
pivot

label,subject_id,hadm_id,stay_id,chartdate,intime,first_careunit,outtime,los,gender,age,...,Resp Alarm - High,Resp Alarm - Low,Respiratory Rate,Temperature,Weight,if_Access Lines - Invasive,if_Cardiovascular,if_Dialysis,if_IABP,if_Impella
0,10000690,25860671,37081114,2150-11-03,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,35.0,8.0,21.473684,36.666667,55.156787,NaN,NaN,NaN,NaN,NaN
1,10000690,25860671,37081114,2150-11-03,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,35.0,8.0,22.000000,36.111111,NaN,NaN,NaN,NaN,NaN,NaN
2,10000690,25860671,37081114,2150-11-04,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,NaN,NaN,20.850000,36.177778,55.156787,NaN,NaN,NaN,NaN,NaN
3,10000690,25860671,37081114,2150-11-04,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,NaN,NaN,21.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10000690,25860671,37081114,2150-11-05,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,32.5,8.0,24.523810,36.791667,NaN,1.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441380,19999442,26785317,32336619,2148-11-24,2148-11-19 14:23:43,Surgical Intensive Care Unit (SICU),2148-11-26 13:12:15,6.950370,M,43,...,35.0,8.0,14.428571,36.833333,NaN,NaN,NaN,NaN,NaN,NaN
441381,19999625,25304202,31070865,2139-10-11,2139-10-10 19:18:00,Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-11 18:21:28,0.960741,M,82,...,NaN,NaN,18.111111,36.877778,50.394071,NaN,NaN,NaN,NaN,NaN
441382,19999828,25744818,36075953,2149-01-10,2149-01-08 18:12:00,Medical Intensive Care Unit (MICU),2149-01-10 13:11:02,1.790995,F,48,...,35.0,8.0,12.363636,36.694444,67.700000,NaN,NaN,NaN,NaN,NaN
441383,19999828,25744818,36075953,2149-01-09,2149-01-08 18:12:00,Medical Intensive Care Unit (MICU),2149-01-10 13:11:02,1.790995,F,48,...,35.0,8.0,15.000000,36.811111,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
pivot.isna().sum()*100/len(pivot)

label
subject_id                           0.000000
hadm_id                              0.000000
stay_id                              0.000000
chartdate                            0.000000
intime                               0.000000
first_careunit                       0.000000
outtime                              0.000000
los                                  0.000000
gender                               0.000000
age                                  0.000000
stay_days                            0.000000
anchor_year                          0.000000
anchor_year_group                    0.000000
admission_type                       0.000000
admission_location                   0.000000
insurance                            0.000000
language                             0.000000
marital_status                       0.000000
race                                 0.000000
icu_types                            0.000000
assigned_intime                      0.000000
assigned_outtime            

### Handle physiological outliers
First find out how many extreme outliers. Extreme here means physiologically impossible.
This is not fixed, so remaining outliers should be dealt at the algorithm level at EDA

In [14]:
len(pivot[(pivot['ART BP Mean']<=0) | (pivot['ART BP Mean']>=1000)])

37

In [15]:
len(pivot[(pivot['Arterial Blood Pressure mean']<=0) | (pivot['Arterial Blood Pressure mean']>=1000)])

102

In [16]:
len(pivot[(pivot['Heart Rate']<=0) | (pivot['Heart Rate']>=1000)])

6

In [17]:
len(pivot[(pivot['Heart rate Alarm - High']<=0) | (pivot['Heart rate Alarm - High']>=1000)])

156

In [18]:
len(pivot[(pivot['Height']<=0) | (pivot['Height']>=300)])

0

In [19]:
len(pivot[(pivot['Inspired O2 Fraction']<=0) | (pivot['Inspired O2 Fraction']>=100)])

2067

In [20]:
len(pivot[(pivot['Non Invasive Blood Pressure mean']<=0) | (pivot['Non Invasive Blood Pressure mean']>=1000)])

26

In [21]:
len(pivot[(pivot['Resp Alarm - High']<=0) | (pivot['Resp Alarm - High']>=300)])

75

In [22]:
len(pivot[(pivot['Resp Alarm - Low']<=0) | (pivot['Resp Alarm - Low']>=300)])

93

In [23]:
len(pivot[(pivot['Respiratory Rate']<=0) | (pivot['Respiratory Rate']>=300)])

71

In [24]:
len(pivot[(pivot['Temperature']<=20) | (pivot['Temperature']>=50)])

323

In [25]:
len(pivot[(pivot['Weight']<=0) | (pivot['Weight']>=700)])

79

In [26]:
pivot.Weight.max()

np.float64(14079.4049616)

In [27]:
pivot.loc[(pivot['ART BP Mean'] <= 0) | (pivot['ART BP Mean'] >= 1000), 'ART BP Mean'] = pd.NA
pivot.loc[(pivot['Arterial Blood Pressure mean']<=0) | (pivot['Arterial Blood Pressure mean']>=1000),'Arterial Blood Pressure mean'] = pd.NA
pivot.loc[(pivot['Heart Rate']<=0) | (pivot['Heart Rate']>=1000),'Heart Rate'] = pd.NA
pivot.loc[(pivot['Heart rate Alarm - High']<=0) | (pivot['Heart rate Alarm - High']>=1000),'Heart rate Alarm - High'] = pd.NA
pivot.loc[(pivot['Height']<=0) | (pivot['Height']>=300),'Height'] = pd.NA
pivot.loc[(pivot['Inspired O2 Fraction']<=0) | (pivot['Inspired O2 Fraction']>=100),'Inspired O2 Fraction'] = pd.NA
pivot.loc[(pivot['Non Invasive Blood Pressure mean']<=0) | (pivot['Non Invasive Blood Pressure mean']>=1000),'Non Invasive Blood Pressure mean'] = pd.NA
pivot.loc[(pivot['Resp Alarm - High']<=0) | (pivot['Resp Alarm - High']>=300),'Resp Alarm - High'] = pd.NA
pivot.loc[(pivot['Resp Alarm - Low']<=0) | (pivot['Resp Alarm - Low']>=300),'Resp Alarm - Low'] = pd.NA
pivot.loc[(pivot['Respiratory Rate']<=0) | (pivot['Respiratory Rate']>=300),'Respiratory Rate'] = pd.NA
pivot.loc[(pivot['Temperature']<=20) | (pivot['Temperature']>=50),'Temperature'] = pd.NA
pivot.loc[(pivot['Weight']<=0) | (pivot['Weight']>=700),'Weight'] = pd.NA

### Check for sparse column

In [30]:
cols = ['ART BP Mean', 'Arterial Blood Pressure mean','Family Meeting', 'Heart Rate', 
        'Heart rate Alarm - High', 'Height',
        'Inspired O2 Fraction','Non Invasive Blood Pressure mean',
        'Resp Alarm - High', 'Resp Alarm - Low', 'Respiratory Rate',
        'Temperature', 'Weight']

pivot[cols].count()

label
ART BP Mean                          16946
Arterial Blood Pressure mean        145926
Family Meeting                         669
Heart Rate                          439457
Heart rate Alarm - High             319700
Height                                   8
Inspired O2 Fraction                175078
Non Invasive Blood Pressure mean    343058
Resp Alarm - High                   314221
Resp Alarm - Low                    314148
Respiratory Rate                    436461
Temperature                         393590
Weight                              131473
dtype: int64

In [31]:
pivot = pivot.drop(columns=['Height'])

### Handle missing values for numeric

For missing values for chart events, we are assuming Missing At Random (MAR) because there is significance when a test is not conducted. We use the MICE algorithm as specified in this article
https://pmc.ncbi.nlm.nih.gov/articles/PMC8499698/

**MICE** is a robut algorithm for imputing missing values for numerical data in medical datasets for Missing At Random (MAR) through simulation and choosing the best values

The Python package miceforest is used

In [32]:
# Function to perform MICE algorithm
def MICE_impute(df, cols):
    df_numeric = df[cols]
    
    kernel = mf.ImputationKernel(
        df_numeric,
        random_state=5188
    )
    
    # MICE imputation for 20 datasets
    kernel.mice(20)
    
    # Retrieve the imputed dataset
    imputed_df = kernel.complete_data()

    return imputed_df

In [33]:
# Call the function to perform MICE impute
cols = ['ART BP Mean', 'Arterial Blood Pressure mean',
        'Family Meeting', 'Heart Rate', 'Heart rate Alarm - High',
        'Inspired O2 Fraction','Non Invasive Blood Pressure mean',
        'Resp Alarm - High', 'Resp Alarm - Low', 'Respiratory Rate',
        'Temperature', 'Weight']
imputed_df = MICE_impute(pivot, cols)
# Combine the data back to pivot
pivot = pd.concat([pivot[['subject_id', 'hadm_id', 'stay_id','chartdate','intime', 'first_careunit',
                          'outtime', 'los', 'gender','age', 'anchor_year','stay_days',
                          'anchor_year_group', 'admission_type', 'admission_location',
                          'insurance', 'language', 'marital_status', 'race',
                          'icu_types', 'assigned_intime','High risk (>51) interventions',
                          'Low risk (25-50) interventions','assigned_outtime','if_Access Lines - Invasive',
                          'if_Cardiovascular','if_Dialysis', 'if_IABP', 'if_Impella']], imputed_df], axis=1)

### Handle missing values for categorical.

This missing at random, and when no tests are taken, it is inferred that the test is unnecessary, hence 0.

In [34]:
cols_to_fill = [
    'High risk (>51) interventions',
    'Low risk (25-50) interventions',
    'if_Access Lines - Invasive',
    'if_Cardiovascular',
    'if_Dialysis',
    'if_IABP',
    'if_Impella'
]

pivot.loc[:, cols_to_fill] = pivot.loc[:, cols_to_fill].fillna(0)

In [35]:
# Make sure no missing values
pivot.isna().sum()*100/len(pivot)

label
subject_id                          0.0
hadm_id                             0.0
stay_id                             0.0
chartdate                           0.0
intime                              0.0
first_careunit                      0.0
outtime                             0.0
los                                 0.0
gender                              0.0
age                                 0.0
anchor_year                         0.0
stay_days                           0.0
anchor_year_group                   0.0
admission_type                      0.0
admission_location                  0.0
insurance                           0.0
language                            0.0
marital_status                      0.0
race                                0.0
icu_types                           0.0
assigned_intime                     0.0
High risk (>51) interventions       0.0
Low risk (25-50) interventions      0.0
assigned_outtime                    0.0
if_Access Lines - Invasive        

### Generate total number of chart events and cumulative mean for each subject_id, hadm_id, chartdate based on window of subject_id, hadm_id

In [36]:
pivot['num_chart_events'] = 1
pivot = pivot.sort_values(by=['subject_id', 'hadm_id', 'stay_id','chartdate'])
pivot['Respiratory Alarm Margin High'] = pivot['Resp Alarm - High'] - pivot['Respiratory Rate']
pivot['Respiratory Alarm Margin Low'] = pivot['Respiratory Rate'] - pivot['Resp Alarm - Low']
pivot['Heart rate Alarm Margin High'] = pivot['Heart rate Alarm - High'] - pivot['Heart Rate']


#numeric
pivot['Change in ART BP Mean'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['ART BP Mean'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Arterial Blood Pressure mean'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Arterial Blood Pressure mean'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Family Meeting'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Family Meeting'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Heart Rate'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Heart Rate'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Heart rate Alarm Margin High'] = pivot.groupby(['subject_id', 'hadm_id','stay_id'],as_index=False)['Heart rate Alarm Margin High'].transform(lambda x: (x - x.expanding().mean())).fillna(0)
pivot['Change in Inspired O2 Fraction'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Inspired O2 Fraction'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Non Invasive Blood Pressure mean'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Non Invasive Blood Pressure mean'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Resp Alarm - High'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Resp Alarm - High'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Resp Alarm - Low'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Resp Alarm - Low'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Respiratory Rate'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Respiratory Rate'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Respiratory Alarm Margin High'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Respiratory Alarm Margin High'].transform(lambda x: (x - x.expanding().mean())).fillna(0)
pivot['Change in Respiratory Alarm Margin Low'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Respiratory Alarm Margin Low'].transform(lambda x: (x - x.expanding().mean())).fillna(0)
pivot['Change in Temperature'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Temperature'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
pivot['Change in Weight'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id',],as_index=False)['Weight'].transform(lambda x: (x - x.expanding().mean())/x.expanding().mean()).fillna(0)
#numeric 24h
pivot['Last 24H ART BP Mean'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['ART BP Mean'].transform(lambda x: x).fillna(0)
pivot['Last 24H Arterial Blood Pressure mean'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Arterial Blood Pressure mean'].transform(lambda x: x).fillna(0)
pivot['Last 24H Family Meeting'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Family Meeting'].transform(lambda x: x).fillna(0)
pivot['Last 24H Heart Rate'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Heart Rate'].transform(lambda x: x).fillna(0)
pivot['Last 24H Heart rate Alarm Margin High'] = pivot.groupby(['subject_id', 'hadm_id','stay_id'],as_index=False)['Heart rate Alarm Margin High'].transform(lambda x: x).fillna(0)
pivot['Last 24H Inspired O2 Fraction'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Inspired O2 Fraction'].transform(lambda x: x).fillna(0)
pivot['Last 24H Non Invasive Blood Pressure mean'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Non Invasive Blood Pressure mean'].transform(lambda x: x).fillna(0)
pivot['Last 24H Respiratory Alarm Margin High'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Respiratory Alarm Margin High'].transform(lambda x: x).fillna(0)
pivot['Last 24H Respiratory Alarm Margin Low'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Respiratory Alarm Margin Low'].transform(lambda x: x).fillna(0)
pivot['Last 24H Respiratory Rate'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Respiratory Rate'].transform(lambda x: x).fillna(0)
pivot['Last 24H Temperature'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Temperature'].transform(lambda x: x).fillna(0)
pivot['Last 24H Weight'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id',],as_index=False)['Weight'].transform(lambda x: x).fillna(0)
pivot['Last 24H if_Access Lines - Invasive'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Access Lines - Invasive'].transform(lambda x: x).fillna(0)
pivot['Last 24H if_Cardiovascular'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Cardiovascular'].transform(lambda x: x).fillna(0)
pivot['Last 24H if_Dialysis'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Dialysis'].transform(lambda x: x).fillna(0)
pivot['Last 24H if_IABP'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_IABP'].transform(lambda x: x).fillna(0)
pivot['Last 24H if_Impella'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Impella'].transform(lambda x: x).fillna(0)
pivot['Last 24H High risk (>51) interventions'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['High risk (>51) interventions'].transform(lambda x: x).fillna(0)
pivot['Last 24H Low risk (25-50) interventions'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Low risk (25-50) interventions'].transform(lambda x: x).fillna(0)

#checkbox
pivot['Change if_Access Lines - Invasive'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Access Lines - Invasive'].transform(lambda x: x - x.expanding().mean()).fillna(0)
pivot['Change if_Cardiovascular'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Cardiovascular'].transform(lambda x: x - x.expanding().mean()).fillna(0)
pivot['Change if_Dialysis'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Dialysis'].transform(lambda x: x - x.expanding().mean()).fillna(0)
pivot['Change if_IABP'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_IABP'].transform(lambda x: x - x.expanding().mean()).fillna(0)
pivot['Change if_Impella'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['if_Impella'].transform(lambda x: x - x.expanding().mean()).fillna(0)
pivot['Change High risk (>51) interventions'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['High risk (>51) interventions'].transform(lambda x: x - x.expanding().mean()).fillna(0)
pivot['Change Low risk (25-50) interventions'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['Low risk (25-50) interventions'].transform(lambda x: x - x.expanding().mean()).fillna(0)

#num events
pivot['num_chart_events'] = pivot.groupby(['subject_id', 'hadm_id', 'stay_id'],as_index=False)['num_chart_events'].transform(lambda x: x.expanding().sum())

In [37]:
pivot.head()

label,subject_id,hadm_id,stay_id,chartdate,intime,first_careunit,outtime,los,gender,age,...,Last 24H if_Impella,Last 24H High risk (>51) interventions,Last 24H Low risk (25-50) interventions,Change if_Access Lines - Invasive,Change if_Cardiovascular,Change if_Dialysis,Change if_IABP,Change if_Impella,Change High risk (>51) interventions,Change Low risk (25-50) interventions
0,10000690,25860671,37081114,2150-11-03,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,10000690,25860671,37081114,2150-11-03,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.500000,0.0
2,10000690,25860671,37081114,2150-11-04,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.333333,0.0
3,10000690,25860671,37081114,2150-11-04,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.500000,0.0
4,10000690,25860671,37081114,2150-11-05,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,1.0,0.0,0.8,0.0,0.0,0.0,0.0,0.400000,0.0


In [38]:
# Make sure no missing values
pivot.isna().sum()*100/len(pivot)

label
subject_id                                    0.0
hadm_id                                       0.0
stay_id                                       0.0
chartdate                                     0.0
intime                                        0.0
first_careunit                                0.0
outtime                                       0.0
los                                           0.0
gender                                        0.0
age                                           0.0
anchor_year                                   0.0
stay_days                                     0.0
anchor_year_group                             0.0
admission_type                                0.0
admission_location                            0.0
insurance                                     0.0
language                                      0.0
marital_status                                0.0
race                                          0.0
icu_types                                   

In [39]:
pivot.to_csv(output_folder+ 'pivot/preprocessed_patients_chartevents.csv', index=False)

## 2. Add Procedures

### Combine procedures with icd details to get text description of the procedures taken by patient

In [41]:
df_train = pd.read_csv(output_folder+ 'pivot/preprocessed_patients_chartevents.csv')
df_icd = pd.read_csv(input_folder+ 'd_icd_procedures.csv')
df_procedures = pd.read_csv(input_folder+ 'procedures_icd.csv')
df_joined = pd.merge(df_procedures, df_icd, on=['icd_code', 'icd_version'], how='inner')
df_proc_joined = pd.merge(df_train, df_joined, on=['subject_id', 'hadm_id'], how='left')
df_proc_joined = df_proc_joined.drop(columns=['icd_version','icd_code','seq_num']).\
                                rename(columns={'chartdate_x':'chartdate','chartdate_y':'proc_date','long_title':'procedure'})

In [42]:
# Filter to procedures done within ICU stay
df_proc_joined = df_proc_joined[(df_proc_joined.proc_date >= df_proc_joined.intime) & (df_proc_joined.proc_date <= df_proc_joined.outtime)]

In [43]:
# Filter to procedures done prior to chartdate to prevent leakage
df_proc_joined['proc_datediff'] = (pd.to_datetime(df_proc_joined['chartdate']) - pd.to_datetime(df_proc_joined['proc_date'])).dt.days
df_proc_joined['proc_datediff'] = np.where(df_proc_joined.proc_datediff < 0, np.nan, df_proc_joined.proc_datediff)

### Create new features related to procedures
Concatenation of all procedures taken by the patient before chartevent (to prevent leakgage) and the number of procedures

In [44]:
# Function to concatenate all the valid procedures
def concat_procedures(group):
    has_valid_procs = (group['proc_datediff'] >= 0).any()
    
    if not has_valid_procs:
        return np.nan
    
    else:
        valid_group = group[group['proc_datediff'] >= 0]
        procedure_list = valid_group['procedure'].astype(str)
        
        return ', '.join(procedure_list)

# Function to count all the valid procedures
def sum_procedures(group):
    has_valid_procs = (group['proc_datediff'] >= 0).any()
    
    if not has_valid_procs:
        return 0
    
    else:
        valid_group = group[group['proc_datediff'] >= 0]
        procedure_list = valid_group['procedure'].astype(str)
        
        return len(procedure_list)

In [45]:
proc_concat = df_proc_joined.groupby(['subject_id', 'hadm_id','chartdate']).apply(concat_procedures,include_groups=False).rename('procedure_concat')
num_proc = df_proc_joined.groupby(['subject_id', 'hadm_id','chartdate']).apply(sum_procedures,include_groups=False).fillna(0).rename('num_procedures')

In [46]:
df_train = pd.merge(df_train, proc_concat, on=['subject_id', 'hadm_id','chartdate'], how='left')
df_train = pd.merge(df_train, num_proc, on=['subject_id', 'hadm_id','chartdate'], how='left')

In [47]:
df_train

,subject_id,hadm_id,stay_id,chartdate,intime,first_careunit,outtime,los,gender,age,...,Last 24H Low risk (25-50) interventions,Change if_Access Lines - Invasive,Change if_Cardiovascular,Change if_Dialysis,Change if_IABP,Change if_Impella,Change High risk (>51) interventions,Change Low risk (25-50) interventions,procedure_concat,num_procedures
0,10000690,25860671,37081114,2150-11-03,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,NaN,NaN
1,10000690,25860671,37081114,2150-11-03,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,0.000000,0.0,0.0,0.0,0.0,-0.500000,0.0,NaN,NaN
2,10000690,25860671,37081114,2150-11-04,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.333333,0.0,NaN,NaN
3,10000690,25860671,37081114,2150-11-04,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,0.000000,0.0,0.0,0.0,0.0,-0.500000,0.0,NaN,NaN
4,10000690,25860671,37081114,2150-11-05,2150-11-02 19:37:00,Medical Intensive Care Unit (MICU),2150-11-06 17:03:17,3.893252,F,86,...,0.0,0.800000,0.0,0.0,0.0,0.0,0.400000,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441380,19999442,26785317,32336619,2148-11-26,2148-11-19 14:23:43,Surgical Intensive Care Unit (SICU),2148-11-26 13:12:15,6.950370,M,43,...,0.0,-0.230769,0.0,0.0,0.0,0.0,-0.307692,0.0,Enteral infusion of concentrated nutritional s...,1.0
441381,19999625,25304202,31070865,2139-10-11,2139-10-10 19:18:00,Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-11 18:21:28,0.960741,M,82,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,NaN,NaN
441382,19999828,25744818,36075953,2149-01-09,2149-01-08 18:12:00,Medical Intensive Care Unit (MICU),2149-01-10 13:11:02,1.790995,F,48,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,NaN,0.0
441383,19999828,25744818,36075953,2149-01-09,2149-01-08 18:12:00,Medical Intensive Care Unit (MICU),2149-01-10 13:11:02,1.790995,F,48,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,NaN,0.0


In [48]:
df_train.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'chartdate', 'intime',
       'first_careunit', 'outtime', 'los', 'gender', 'age', 'anchor_year',
       'stay_days', 'anchor_year_group', 'admission_type',
       'admission_location', 'insurance', 'language', 'marital_status', 'race',
       'icu_types', 'assigned_intime', 'High risk (>51) interventions',
       'Low risk (25-50) interventions', 'assigned_outtime',
       'if_Access Lines - Invasive', 'if_Cardiovascular', 'if_Dialysis',
       'if_IABP', 'if_Impella', 'ART BP Mean', 'Arterial Blood Pressure mean',
       'Family Meeting', 'Heart Rate', 'Heart rate Alarm - High',
       'Inspired O2 Fraction', 'Non Invasive Blood Pressure mean',
       'Resp Alarm - High', 'Resp Alarm - Low', 'Respiratory Rate',
       'Temperature', 'Weight', 'num_chart_events',
       'Respiratory Alarm Margin High', 'Respiratory Alarm Margin Low',
       'Heart rate Alarm Margin High', 'Change in ART BP Mean',
       'Change in Arterial Blood Pressure mean

## 3. Change target to the remaining los from chartdate

In [ ]:
df_train['los'] = (pd.to_datetime(df_train['outtime']) - pd.to_datetime(df_train['chartdate']).dt.normalize()).dt.days
df_train = df_train.rename(columns = {'los':'remaining los'})
df_train.head()

In [ ]:
len(df_train)

In [ ]:
# drop original values, keeping the derived values
df_train = df_train.drop(['ART BP Mean','Arterial Blood Pressure mean','Family Meeting','Change in Family Meeting','Heart Rate',
                    'Heart rate Alarm - High','High risk (>51) interventions','Inspired O2 Fraction',
                    'Low risk (25-50) interventions','Non Invasive Blood Pressure mean','Resp Alarm - High','Resp Alarm - Low',
                    'Respiratory Rate','Temperature','Weight','if_Access Lines - Invasive',
                    'if_Cardiovascular','if_Dialysis','if_IABP','if_Impella'],axis=1)

In [ ]:
# Last check to see if the missing values are expected, or will be dealt in the training notebook
df_train.isna().sum()*100/len(df_train)

In [ ]:
df_train['num_procedures'] = df_train['num_procedures'].fillna(0)

In [ ]:
df_train.to_csv(output_folder+ 'pivot/preprocessed_patients_chartevents_proc.csv', index=False)

## 4. Add in Vasopressor drugs taken
Include the number of any of the listed vasopressor drugs taken by the patient as a feature

In [55]:
df_train = pd.read_csv(output_folder+ 'pivot/preprocessed_patients_chartevents_proc.csv')
df_ie = pd.read_csv(input_folder+ 'inputevents.csv.gz')

In [56]:
# Verify that the list of vasopressor drugs are correct
df_items = pd.read_csv(input_folder+ 'd_items.csv')
df_items[df_items.itemid.isin([221906,221289,221662,222315,221749,221653])]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
280,221289,Epinephrine,Epinephrine,inputevents,Medications,mg,Solution,NaN,NaN
292,221653,Dobutamine,Dobutamine,inputevents,Medications,mg,Solution,NaN,NaN
293,221662,Dopamine,Dopamine,inputevents,Medications,mg,Solution,NaN,NaN
299,221749,Phenylephrine,Phenylephrine,inputevents,Medications,mg,Solution,NaN,NaN
305,221906,Norepinephrine,Norepinephrine,inputevents,Medications,mg,Solution,NaN,NaN
318,222315,Vasopressin,Vasopressin,inputevents,Medications,units,Solution,NaN,NaN


In [57]:
df_ie = df_ie[df_ie.itemid.isin([221906,221289,221662,222315,221749,221653])]
df_ie

,subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,amount,amountuom,...,ordercomponenttypedescription,ordercategorydescription,patientweight,totalamount,totalamountuom,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate
12,10000690,25860671,37081114,17393,2150-11-02 20:00:00,2150-11-02 22:45:00,2150-11-02 20:14:00,221749,5.475664,mg,...,Main order parameter,Continuous Med,55.3,250.0,mL,0,0,ChangeDose/Rate,60.000000,0.600000
14,10000690,25860671,37081114,17393,2150-11-02 22:45:00,2150-11-02 23:36:00,2150-11-02 23:12:00,221749,1.410112,mg,...,Main order parameter,Continuous Med,55.3,250.0,mL,0,0,ChangeDose/Rate,54.524338,0.500000
16,10000690,25860671,37081114,17393,2150-11-02 23:36:00,2150-11-03 00:35:00,2150-11-03 00:42:00,221749,1.305181,mg,...,Main order parameter,Continuous Med,55.3,250.0,mL,0,0,ChangeDose/Rate,53.114223,0.400000
18,10000690,25860671,37081114,17393,2150-11-03 00:35:00,2150-11-03 01:07:00,2150-11-03 00:43:00,221749,0.530864,mg,...,Main order parameter,Continuous Med,55.3,250.0,mL,0,0,ChangeDose/Rate,51.809040,0.300000
20,10000690,25860671,37081114,17393,2150-11-03 01:07:00,2150-11-03 02:03:00,2150-11-03 02:07:00,221749,0.619409,mg,...,Main order parameter,Continuous Med,55.3,250.0,mL,0,0,Stopped,51.278179,0.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10953625,19999840,21033226,38978960,93123,2164-09-17 07:18:00,2164-09-17 09:10:00,2164-09-17 10:17:00,221906,1.479000,mg,...,Main order parameter,Continuous Med,77.5,250.0,mL,0,0,FinishedRunning,1.479000,0.169954
10953635,19999840,21033226,38978960,93123,2164-09-17 09:10:00,2164-09-17 10:00:00,2164-09-17 10:17:00,221906,0.660448,mg,...,Main order parameter,Continuous Med,77.5,250.0,mL,0,0,ChangeDose/Rate,7.999999,0.170000
10953643,19999840,21033226,38978960,93123,2164-09-17 10:00:00,2164-09-17 10:18:00,2164-09-17 10:17:00,221906,0.279871,mg,...,Main order parameter,Continuous Med,77.5,250.0,mL,0,0,ChangeDose/Rate,7.338842,0.200000
10953647,19999840,21033226,38978960,93123,2164-09-17 10:18:00,2164-09-17 11:37:00,2164-09-17 10:18:00,221906,1.106466,mg,...,Main order parameter,Continuous Med,77.5,250.0,mL,0,0,ChangeDose/Rate,7.058971,0.180000


**Get the aggregated count of vasopressor taken**

In [58]:
df_ie['starttime'] = pd.to_datetime(df_ie['starttime'])
df_train['chartdate'] = pd.to_datetime(df_train['chartdate'])

# Obtain a unique ID for each row in df_train to group by later.
df_train = df_train.reset_index().rename(columns={'index': 'train_row_id'})

# Merge df_train with all drug events from df_ie
merged_df = pd.merge(
    df_train,
    df_ie[['subject_id', 'hadm_id', 'stay_id', 'itemid', 'starttime']],
    on=['subject_id', 'hadm_id', 'stay_id'],
    how='left'
)

# Filter to only drugs given before the chartdate to prevent leakage
filtered_df = merged_df[merged_df['starttime'] < merged_df['chartdate']].copy()

# Group by the original row ID and count the unique drugs to date
vaso_counts_to_date = filtered_df[['train_row_id']].drop_duplicates().copy()
vaso_counts_to_date['Count of Vasopressor'] = 1

# Merge back to original df_train
df_train = pd.merge(
    df_train,
    vaso_counts_to_date,
    on='train_row_id',
    how='left'
)

**Get the count of vasopressor last taken**

In [59]:
# Find the latest starttime for each group
filtered_df['latest_start_time'] = filtered_df.groupby('train_row_id')['starttime'].transform('max')

# Get the count on the latest start time
latest_day_events = filtered_df[filtered_df['starttime'] == filtered_df['latest_start_time']]
latest_day_counts = latest_day_events.groupby('train_row_id')['itemid'].nunique().reset_index(name='Latest Vasopressor')

# Merge back into the original df_train
df_train = pd.merge(
    df_train,
    latest_day_counts,
    on='train_row_id',
    how='left'
)

# Fill NaNs with 0
df_train['Count of Vasopressor'] = df_train['Count of Vasopressor'].fillna(0).astype(int)
df_train['Latest Vasopressor'] = df_train['Latest Vasopressor'].fillna(0).astype(int)

df_train = df_train.drop(columns=['train_row_id'])
df_train[['subject_id', 'chartdate', 'Count of Vasopressor', 'Latest Vasopressor']]

,subject_id,chartdate,Count of Vasopressor,Latest Vasopressor
0,10000690,2150-11-03,1,1
1,10000690,2150-11-03,1,1
2,10000690,2150-11-04,1,1
3,10000690,2150-11-04,1,1
4,10000690,2150-11-05,1,1
...,...,...,...,...
441380,19999442,2148-11-26,0,0
441381,19999625,2139-10-11,0,0
441382,19999828,2149-01-09,0,0
441383,19999828,2149-01-09,0,0


In [60]:
df_train.to_csv(output_folder+ 'pivot/preprocessed_patients_chartevents_proc_vaso.csv', index=False)

## 4. Add in RTT (Renal Replacement Therapy) related treatment
Add in whether a patient has undergone rrt as a feature

In [71]:
df_train = pd.read_csv(output_folder+ 'pivot/preprocessed_patients_chartevents_proc_vaso.csv')

In [72]:
df_pe = pd.read_csv(input_folder+ 'procedureevents.csv.gz')

In [73]:
# Verify that the list of rtt treatments are correct
df_items = pd.read_csv(input_folder+ 'd_items.csv')
df_items[df_items.itemid.isin([225802,225803,225804,225805,225806,225441])]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
1299,225441,Hemodialysis,Hemodialysis,procedureevents,4-Procedures,NaN,Processes,NaN,NaN
1542,225802,Dialysis - CRRT,Dialysis - CRRT,procedureevents,Dialysis,NaN,Processes,NaN,NaN
1543,225803,Dialysis - CVVHD,Dialysis - CVVHD,procedureevents,Dialysis,NaN,Processes,NaN,NaN
1544,225805,Peritoneal Dialysis,Peritoneal Dialysis,procedureevents,Dialysis,NaN,Processes,NaN,NaN
1545,225806,Volume In (PD),Volume In (PD),chartevents,Dialysis,mL,Numeric,NaN,NaN


In [74]:
df_pe = df_pe[df_pe.itemid.isin([225802,225803,225804,225805,225806,225441])]
df_pe

,subject_id,hadm_id,stay_id,caregiver_id,starttime,endtime,storetime,itemid,value,valueuom,...,orderid,linkorderid,ordercategoryname,ordercategorydescription,patientweight,isopenbag,continueinnextdept,statusdescription,originalamount,originalrate
233,10003637,28317408,32824762,25804.0,2150-05-20 20:48:00,2150-05-22 11:01:00,2150-05-22 15:58:00,225802,2293.0,min,...,4689407,4689407,Dialysis,ContinuousProcess,88.0,1,0,FinishedRunning,2293.0,1
255,10004235,24181354,34100191,3178.0,2196-02-26 01:00:00,2196-02-27 16:27:00,2196-02-27 16:42:00,225802,2367.0,min,...,7940238,7940238,Dialysis,ContinuousProcess,127.0,1,0,FinishedRunning,2367.0,1
537,10005593,26835370,34389119,96640.0,2125-06-27 18:15:00,2125-06-27 21:33:00,2125-06-27 22:04:00,225441,198.0,min,...,4469291,4469291,Continuous Procedures,ContinuousProcess,67.0,1,0,FinishedRunning,198.0,1
643,10006131,27849136,30177745,NaN,2143-03-30 03:00:00,2143-03-31 17:56:00,2143-03-31 17:56:41.563,225441,2336.0,min,...,4794030,4794030,Continuous Procedures,ContinuousProcess,78.8,1,0,FinishedRunning,2336.0,1
688,10007818,22987108,32359580,6823.0,2146-06-22 21:10:00,2146-06-24 09:58:00,2146-06-24 10:06:00,225802,2208.0,min,...,8723340,8723340,Dialysis,ContinuousProcess,86.2,1,0,FinishedRunning,2208.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
807484,19989783,26984195,32761676,32126.0,2130-07-27 08:42:00,2130-07-27 12:12:00,2130-07-27 09:19:00,225441,3.5,hour,...,4380426,4380426,Continuous Procedures,ContinuousProcess,101.0,0,0,FinishedRunning,3.5,1
807490,19989783,26984195,32761676,32207.0,2130-07-23 13:45:00,2130-07-23 17:16:00,2130-07-24 13:50:00,225441,211.0,min,...,5250534,5250534,Continuous Procedures,ContinuousProcess,101.0,1,0,FinishedRunning,211.0,1
807495,19989783,26984195,32761676,33148.0,2130-07-13 00:00:00,2130-07-19 15:00:00,2130-07-19 17:35:00,225802,9540.0,min,...,6614496,6614496,Dialysis,ContinuousProcess,101.0,1,0,FinishedRunning,9540.0,1
807498,19989783,26984195,32761676,67563.0,2130-07-25 08:19:00,2130-07-25 11:49:00,2130-07-25 08:19:00,225441,3.5,hour,...,2143256,2143256,Continuous Procedures,ContinuousProcess,101.0,0,0,FinishedRunning,3.5,1


In [75]:
df_pe.endtime = pd.to_datetime(df_pe.endtime)
# Get the max endtime when a rrt is taken
df_pe = df_pe.groupby(['subject_id', 'hadm_id', 'stay_id'])['endtime'].max().reset_index(name='is RRT')
df_pe

,subject_id,hadm_id,stay_id,is RRT
0,10003637,28317408,32824762,2150-05-22 11:01:00
1,10004235,24181354,34100191,2196-02-27 16:27:00
2,10005593,26835370,34389119,2125-06-27 21:33:00
3,10006131,27849136,30177745,2143-03-31 17:56:00
4,10007818,22987108,32359580,2146-07-09 12:02:00
...,...,...,...,...
4467,19975248,27871798,37029190,2157-08-09 17:45:00
4468,19979982,25466463,36681043,2135-08-16 15:31:00
4469,19986880,28386154,32959861,2185-08-10 02:48:00
4470,19989783,24282820,32711376,2130-08-06 11:37:00


In [76]:
df_train = pd.merge(df_train, df_pe,
                    on=['subject_id', 'hadm_id', 'stay_id'],
                    how='left')

In [77]:
# Convert the max endtime to binary
df_train['is RRT'] =  (df_train['is RRT'] <= df_train.chartdate).astype(int)

In [78]:
df_train.to_csv(output_folder+ 'pivot/train_los.csv', index=False)

In [79]:
df_train.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'chartdate', 'intime',
       'first_careunit', 'outtime', 'remaining los', 'gender', 'age',
       'anchor_year', 'stay_days', 'anchor_year_group', 'admission_type',
       'admission_location', 'insurance', 'language', 'marital_status', 'race',
       'icu_types', 'assigned_intime', 'assigned_outtime', 'num_chart_events',
       'Respiratory Alarm Margin High', 'Respiratory Alarm Margin Low',
       'Heart rate Alarm Margin High', 'Change in ART BP Mean',
       'Change in Arterial Blood Pressure mean', 'Change in Heart Rate',
       'Change in Heart rate Alarm Margin High',
       'Change in Inspired O2 Fraction',
       'Change in Non Invasive Blood Pressure mean',
       'Change in Resp Alarm - High', 'Change in Resp Alarm - Low',
       'Change in Respiratory Rate', 'Change in Respiratory Alarm Margin High',
       'Change in Respiratory Alarm Margin Low', 'Change in Temperature',
       'Change in Weight', 'Last 24H ART BP Mean',
      